In [ ]:
!pip install -q tensorflow opencv-python pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import cv2
import os

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from google.colab import files

In [ ]:
uploaded = files.upload()

Saving faces.zip to faces.zip


In [ ]:
!unzip -q faces.zip
!ls

faces.zip  sample_data	Train  train.csv


In [ ]:
df = pd.read_csv("train.csv")

df["filename"] = "Train/" + df["ID"]

df.head()

,ID,Class,filename
0,377.jpg,MIDDLE,Train/377.jpg
1,17814.jpg,YOUNG,Train/17814.jpg
2,21283.jpg,MIDDLE,Train/21283.jpg
3,16496.jpg,YOUNG,Train/16496.jpg
4,4487.jpg,MIDDLE,Train/4487.jpg


In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [ ]:
def detect_face(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5
    )

    if len(faces) == 0:
        return img

    x,y,w,h = faces[0]

    face = img[y:y+h, x:x+w]

    return face

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=25,
    zoom_range=0.25,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1,
    validation_split=0.2
)

In [ ]:
train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    x_col="filename",
    y_col="Class",
    target_size=(128,128),
    class_mode="categorical",
    batch_size=32,
    subset="training"
)

val_generator = datagen.flow_from_dataframe(
    dataframe=df,
    x_col="filename",
    y_col="Class",
    target_size=(128,128),
    class_mode="categorical",
    batch_size=32,
    subset="validation"
)

Found 15925 validated image filenames belonging to 3 classes.
Found 3981 validated image filenames belonging to 3 classes.


In [ ]:
base_model = MobileNetV2(
    input_shape=(128,128,3),
    include_top=False,
    weights="imagenet"
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(512, activation="relu")(x)

x = Dropout(0.5)(x)

output = Dense(3, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 2,915,395 (11.12 MB)

 Trainable params: 1,863,491 (7.11 MB)

 Non-trainable params: 1,051,904 (4.01 MB)

In [ ]:
checkpoint = ModelCheckpoint(
    "best_age_model.h5",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

earlystop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3
)

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=[checkpoint, earlystop, reduce_lr]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 566ms/step - accuracy: 0.5788 - loss: 0.9951
Epoch 1: val_accuracy improved from -inf to 0.63904, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 347s 679ms/step - accuracy: 0.5788 - loss: 0.9949 - val_accuracy: 0.6390 - val_loss: 1.0156 - learning_rate: 1.0000e-04
Epoch 2/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - accuracy: 0.6667 - loss: 0.7674
Epoch 2: val_accuracy improved from 0.63904 to 0.66215, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 360s 723ms/step - accuracy: 0.6667 - loss: 0.7673 - val_accuracy: 0.6621 - val_loss: 0.9756 - learning_rate: 1.0000e-04
Epoch 3/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 565ms/step - accuracy: 0.6958 - loss: 0.7012
Epoch 3: val_accuracy improved from 0.66215 to 0.69430, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 336s 675ms/step - accuracy: 0.6958 - loss: 0.7012 - val_accuracy: 0.6943 - val_loss: 0.8048 - learning_rate: 1.0000e-04
Epoch 4/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 559ms/step - accuracy: 0.7149 - loss: 0.6714
Epoch 4: val_accuracy improved from 0.69430 to 0.69606, saving model to best_age_model.h5


...

498/498 ━━━━━━━━━━━━━━━━━━━━ 373s 668ms/step - accuracy: 0.7812 - loss: 0.5228 - val_accuracy: 0.7589 - val_loss: 0.6005 - learning_rate: 3.0000e-05
Epoch 13/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 559ms/step - accuracy: 0.7813 - loss: 0.5197
Epoch 13: val_accuracy did not improve from 0.75885
498/498 ━━━━━━━━━━━━━━━━━━━━ 332s 667ms/step - accuracy: 0.7813 - loss: 0.5197 - val_accuracy: 0.7491 - val_loss: 0.6066 - learning_rate: 3.0000e-05
Epoch 14/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 570ms/step - accuracy: 0.7784 - loss: 0.5253
Epoch 14: val_accuracy did not improve from 0.75885
498/498 ━━━━━━━━━━━━━━━━━━━━ 339s 680ms/step - accuracy: 0.7784 - loss: 0.5253 - val_accuracy: 0.7425 - val_loss: 0.6334 - learning_rate: 3.0000e-05
Epoch 15/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 607ms/step - accuracy: 0.7859 - loss: 0.5129
Epoch 15: val_accuracy did not improve from 0.75885
498/498 ━━━━━━━━━━━━━━━━━━━━ 364s 732ms/step - accuracy: 0.7859 - loss: 0.5129 - val_accuracy: 0.7576 - val_loss: 0.5674 - learnin

498/498 ━━━━━━━━━━━━━━━━━━━━ 333s 670ms/step - accuracy: 0.7990 - loss: 0.4915 - val_accuracy: 0.7624 - val_loss: 0.5917 - learning_rate: 3.0000e-05
Epoch 17/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 555ms/step - accuracy: 0.7995 - loss: 0.4879
Epoch 17: val_accuracy did not improve from 0.76237
498/498 ━━━━━━━━━━━━━━━━━━━━ 330s 663ms/step - accuracy: 0.7995 - loss: 0.4879 - val_accuracy: 0.7491 - val_loss: 0.5915 - learning_rate: 3.0000e-05
Epoch 18/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 562ms/step - accuracy: 0.7966 - loss: 0.4900
Epoch 18: val_accuracy improved from 0.76237 to 0.76765, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 362s 728ms/step - accuracy: 0.7966 - loss: 0.4899 - val_accuracy: 0.7676 - val_loss: 0.5691 - learning_rate: 3.0000e-05
Epoch 19/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 556ms/step - accuracy: 0.8005 - loss: 0.4833
Epoch 19: val_accuracy did not improve from 0.76765
498/498 ━━━━━━━━━━━━━━━━━━━━ 331s 664ms/step - accuracy: 0.8005 - loss: 0.4833 - val_accuracy: 0.7646 - val_loss: 0.5482 - learning_rate: 9.0000e-06
Epoch 20/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.8064 - loss: 0.4674
Epoch 20: val_accuracy improved from 0.76765 to 0.77995, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 335s 673ms/step - accuracy: 0.8065 - loss: 0.4674 - val_accuracy: 0.7800 - val_loss: 0.5408 - learning_rate: 9.0000e-06
Epoch 21/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 563ms/step - accuracy: 0.8046 - loss: 0.4748
Epoch 21: val_accuracy did not improve from 0.77995
498/498 ━━━━━━━━━━━━━━━━━━━━ 363s 728ms/step - accuracy: 0.8046 - loss: 0.4748 - val_accuracy: 0.7764 - val_loss: 0.5531 - learning_rate: 9.0000e-06
Epoch 22/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 578ms/step - accuracy: 0.8048 - loss: 0.4723
Epoch 22: val_accuracy did not improve from 0.77995
498/498 ━━━━━━━━━━━━━━━━━━━━ 361s 685ms/step - accuracy: 0.8048 - loss: 0.4723 - val_accuracy: 0.7734 - val_loss: 0.5494 - learning_rate: 9.0000e-06
Epoch 23/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 568ms/step - accuracy: 0.8121 - loss: 0.4620
Epoch 23: val_accuracy improved from 0.77995 to 0.78171, saving model to best_age_model.h5


498/498 ━━━━━━━━━━━━━━━━━━━━ 339s 682ms/step - accuracy: 0.8121 - loss: 0.4620 - val_accuracy: 0.7817 - val_loss: 0.5458 - learning_rate: 9.0000e-06
Epoch 24/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 558ms/step - accuracy: 0.8103 - loss: 0.4597
Epoch 24: val_accuracy did not improve from 0.78171
498/498 ━━━━━━━━━━━━━━━━━━━━ 332s 666ms/step - accuracy: 0.8103 - loss: 0.4597 - val_accuracy: 0.7729 - val_loss: 0.5489 - learning_rate: 2.7000e-06
Epoch 25/25
498/498 ━━━━━━━━━━━━━━━━━━━━ 0s 559ms/step - accuracy: 0.8100 - loss: 0.4658
Epoch 25: val_accuracy did not improve from 0.78171
498/498 ━━━━━━━━━━━━━━━━━━━━ 332s 667ms/step - accuracy: 0.8100 - loss: 0.4658 - val_accuracy: 0.7687 - val_loss: 0.5698 - learning_rate: 2.7000e-06


In [ ]:
model.save("age_model.h5")